1) Набор данных, постановка задачи 

Входных параметров, по которым нам нужно построить модель, будет три – показания 
датчика обводненности Water Cut, и наблюдаемые расход и плотность. Загрузим данные и 
соберем наш датасет. 

1) Загрузите данные. Создайте набор для обучения X, y.

In [2]:
import pandas as pd 
# Load the dataset 
dataset = pd.read_excel('3_inch_3_phase_benchmark_normalised_dd.xlsx') 

# Номера нужных столбцов 
WaterCut = 3            # Показания датчика обводненности
MFR_obs = 7             # наблюдаемый расход 
DD_obs = 9              # наблюдаемая плотность 
 
MFR_err = 10            # ошибка измерения расхода 
# DD_err = 12           # ошибка измерения плотности 
 
# Начинаем с этой строки 
row_start = 3 
 
# Вход и выход модели 
X = dataset.iloc[row_start:, [WaterCut, MFR_obs, DD_obs]].values.astype(float) 
y = dataset.iloc[row_start:, [MFR_err]].values.astype(float)

В итоге мы имеем задачу регрессии, три входных переменных и одну выходную, значения 
которой нашей модели надо научиться прогнозировать. 

2) Разделим данные на тренировочные и тестовые с помощью train_test_split. 

In [3]:
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import PolynomialFeatures 
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_absolute_error 
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Давайте начнем с того, что возьмем простую модель линейной регрессии и попытаемся 
решить нашу задачу. 

In [4]:
regression_model = LinearRegression() 
regression_model.fit(X_train, y_train) 
y_predicted = regression_model.predict(X_test) 
mae = mean_absolute_error(y_test, y_predicted) 
print('Mean absolute error:', mae) 

Mean absolute error: 4.867569879569124


Средняя ошибка составляет 4.9, что, конечно, много. Мы предсказываем значения 
погрешности, которые сами по себе небольшие, иногда почти нулевые, а делаем это с mae = 4.9.  

Конечно, модель, которую мы построили, является слишком простой, она не смогла 
хорошо решить нашу задачу. Давайте заодно вспомним, как можно улучшить простую 
модель линейной регрессии в нашем случае. 

Сейчас у нас всего 3 входных признака, и мы строим модель вида 𝑦 = 𝜃0 + 𝜃1𝑥1 + 𝜃2𝑥2 +𝜃3𝑥3 

3) Используя функцию Polynomial Features, добавим признаки 

In [5]:
poly = PolynomialFeatures(3) 
X_train_ex = poly.fit_transform(X_train) 
X_test_ex = poly.transform(X_test) 

И построим теперь нелинейную модель вида 𝑦 = 𝜃0 +𝜃1𝑥1+𝜃2𝑥2 +𝜃3𝑥3 +𝜃4𝑥12 + 𝜃5𝑥1𝑥2 + 𝜃6𝑥1𝑥3 +𝜃7𝑥2^2 +𝜃8𝑥2𝑥3 +𝜃9𝑥3^2. 

In [6]:
regression_model = LinearRegression() 
regression_model.fit(X_train_ex, y_train) 
y_predicted = regression_model.predict(X_test_ex) 

4) Какая сейчас точность? 
5) Почему мы не беспокоились о нормализации данных? 

In [7]:
mae = mean_absolute_error(y_test, y_predicted)
print('MAE:', mae)

MAE: 1.8756805463934212


Точность стало получше, так как обучили кривую поверхность

5)Потому что линейная регрессия НЕ зависит от масштаба признаков, если:
*мы НЕ используем регуляризацию,
*мы работаем с аналитическим решением (метод наименьших квадратов).

Линейная регрессия ищет решение через формулу: O = (X^T*X)^-1*X^T*y

Это решение масштабно-инвариантно — если один признак умножить на 10, соответствующий коэффициент θ просто уменьшится в 10 раз.

Итак, мы использовали простую модель линейной регрессии с аугментацией признаков. 
Это на самом деле простое, быстрое, но как Вы видите достаточно эффективное решение.  
Если же мы все-таки хотим еще улучшить наш результат, то стоит попробовать другие 
модели. И хотя другие линейные модели с расширением признаком вполне тоже могут 
использоваться  

2. Применение метода Random Forest для решения задачи 

Ссылка [https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html]

Применим для нашей задачи [метод случайного леса] с настройками по умолчанию. Даже в 
таком виде он дает отличный результат.

In [8]:
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import GridSearchCV 

RF = RandomForestRegressor(random_state=42) 
RF.fit(X_train, y_train.ravel()) 
y_predicted = RF.predict(X_test) 
mae = mean_absolute_error(y_test, y_predicted) 
print('Mean absolute error:', mae)

Mean absolute error: 1.2720706385176217


7) Какую точность Вы получили? 

Точность: 1.2720706385176217

Чтобы еще улучшить этот результат, нужно подбирать параметры модели с помощью 
решетчатого поиска и кросс-валидации.  

8) Как правильно построить решетку значений параметров для решетчатого 
поиска? 
9) Какие параметры у модели Random Forest есть?  
10) Каковы их значения по умолчанию? 

In [9]:
params_RF = {'n_estimators':[100, 500, 1500, 2500], 
'max_depth':[ None, 3, 10] 
} 

Воспользуемся также встроенным в sklearn алгоритмом решетчатого поиска вместе с 
кросс-валидацией GridSearchCV. Следующий код выполняется долго. По умолчанию 
кросс-валидация выполняется по 5 папкам. 

In [10]:
grid_cv = GridSearchCV(estimator=RF, param_grid=params_RF, scoring='neg_mean_absolute_error') 
grid_cv.fit(X_train, y_train.ravel()) 

,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None, 3, ...], 'n_estimators': [100, 500, ...]}"
,scoring,'neg_mean_absolute_error'
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,2500


11) Сколько моделей обучается при выполнении этого кода?

При решетке, которая приведена выше в params_RF изначальное всего комбинаций в 12 штук, т.к. 4 значения * 3 значения = 12 комбинаций
Далее, так как GridSearchCV использует 5-fold кросс-валидацию по умолчанию, а значит, 12 штук * 5 фолдов = 60 обученных моделей.

Возьмем теперь лучшие найденные параметры и еще раз обучим нашу модель.   

In [11]:
grid_cv.best_params_ 

RF = RandomForestRegressor(max_depth=None, n_estimators=2500,  random_state=42) 
RF.fit(X_train, y_train.ravel()) 
y_predicted = RF.predict(X_test) 
mae = mean_absolute_error(y_test, y_predicted) 
print('Mean absolute error:', mae) 

Mean absolute error: 1.2415056984007409


12) Какую точность мы получили? 

Mean absolute error: 1.2415056984007409

13) Почему и здесь не встал вопрос о нормализации данных? 

Потому нормализация данных не применяется в дерево решений, так как не получится повлиять на логику делений порогов вида x < value

Нормализация нужна для: линейных моделей, моделей, основанных на расстояниях (kNN), нейронных сетей, SVM.
Но не важна для:Decision Tree, Random Forest, Gradient Boosting

14) Какое расширение решетки Вы бы рекомендовали в связи с найденными 
оптимальными параметрами? 

Можно было бы применить следующую решетку, т.к. при n_estimators = 2500 и max_depth = None показал лучший результат

А также, чтобы избежать переобучения добавляем дополнительные параметры, где из них max_feature делает более разнообразные деревья

In [ ]:
params_RF = {
    'n_estimators': [2000, 3000, 4000],
    'max_depth': [None, 20, 50],
    'min_samples_leaf': [1, 2, 3, 5],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', 1.0]
}

15) Какова идея, суть алгоритма случайного леса?

Построить много разнообразных деревьев, каждая из которых переобучается/обучается по-своему, но их общее среднее даёт устойчивое и точное предсказание.

3. Применение метода Gradient Boosting для решения задачи

Аналогично, как при использовании случайного леса,  

16) примените метод градиентного бустинга с настройками по умолчанию к 
нашему набору данных. 

In [12]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

GB = GradientBoostingRegressor(random_state=42)
GB.fit(X_train, y_train.ravel())
y_pred = GB.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(mae)


1.4523243916397997


17) Какую точность мы получили? 

1.4523243916397997

18) Какие параметры есть у градиентного бустинга? 

🔹 n_estimators

Сколько деревьев в ансамбле (обычно 100–500)

🔹 learning_rate (очень важен!)

Задаёт, насколько сильно каждое дерево влияет на модель.
Маленький → точнее, но нужно больше деревьев.

🔹 max_depth

Глубина каждого дерева
(в отличие от RF, здесь деревья мелкие: 2–5 уровней).

🔹 subsample

Какая доля данных используется для обучения каждого дерева
(например, 0.8).

🔹 min_samples_split / min_samples_leaf

Параметры для контроля переобучения.

🔹 n_estimators

Сколько деревьев в ансамбле (обычно 100–500)

🔹 learning_rate (очень важен!)

Задаёт, насколько сильно каждое дерево влияет на модель.
Маленький → точнее, но нужно больше деревьев.

🔹 max_depth

Глубина каждого дерева
(в отличие от RF, здесь деревья мелкие: 2–5 уровней).

🔹 subsample

Какая доля данных используется для обучения каждого дерева
(например, 0.8).

🔹 min_samples_split / min_samples_leaf

Параметры для контроля переобучения.

19) Какие у них значения по умолчанию? 

Для GradientBoostingRegressor:

Параметр	Значение по умолчанию
n_estimators	100
learning_rate	0.1
max_depth	3
subsample	1.0
criterion	"friedman_mse"
loss	"squared_error"

20) Создайте решетку параметров, обязательно включите туда learning_rate. 

In [13]:
params_GB = {
    'n_estimators': [200, 400, 800],
    'learning_rate': [0.1, 0.05, 0.02],
    'max_depth': [2, 3, 4],
    'subsample': [1.0, 0.8]
}

# Модель градиентного бустинга
GB = GradientBoostingRegressor(random_state=42)

# GridSearchCV
grid_gb = GridSearchCV(
    estimator=GB,
    param_grid=params_GB,
    scoring='neg_mean_absolute_error',
    cv=5,          # 5-fold кросс-валидация
    n_jobs=-1      # Используем все ядра CPU
)

# Обучение
grid_gb.fit(X_train, y_train.ravel())

# Лучшая комбинация параметров
print("Лучшие параметры:", grid_gb.best_params_)

# Лучшая модель
best_gb = grid_gb.best_estimator_

# Предсказание
y_pred = best_gb.predict(X_test)

# Точность
mae = mean_absolute_error(y_test, y_pred)
print("MAE лучшей модели:", mae)

Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 800, 'subsample': 0.8}
MAE лучшей модели: 1.154581098084783


21) За что отвечает параметр learning_rate?

Он задаёт шаг, с которым новое дерево исправляет ошибки прежних.

Если learning_rate маленький:
    каждое дерево делает маленькое изменение
    нужно больше деревьев
    модель получается более точной и менее склонной к переобучению

Если learning_rate большой:
    модель учится быстро, но легко переобучается

Аналогия: шаг в градиентном спуске
Большой шаг → быстро, но может перепрыгнуть минимум.
Маленький шаг → медленно, но точно.

22) Попробуйте добиться mae, меньшей 1.1. Используйте больше параметров в 
Вашей решетке.  

In [14]:
params_GB = {
    'n_estimators': [500, 1000, 1500],
    'learning_rate': [0.1, 0.05, 0.03, 0.01],
    'max_depth': [3, 4],
    'subsample': [0.8, 1.0],
    'min_samples_leaf': [1, 3, 5]
}

# Наша решётка параметров
params_GB = {
    'n_estimators': [200, 400, 800],
    'learning_rate': [0.1, 0.05, 0.02],
    'max_depth': [2, 3, 4],
    'subsample': [1.0, 0.8]
}

# Модель градиентного бустинга
GB = GradientBoostingRegressor(random_state=42)

# GridSearchCV
grid_gb = GridSearchCV(
    estimator=GB,
    param_grid=params_GB,
    scoring='neg_mean_absolute_error',
    cv=5,          # 5-fold кросс-валидация
    n_jobs=-1      # Используем все ядра CPU
)

# Обучение
grid_gb.fit(X_train, y_train.ravel())

# Лучшая комбинация параметров
print("Лучшие параметры:", grid_gb.best_params_)

# Лучшая модель
best_gb = grid_gb.best_estimator_

# Предсказание
y_pred = best_gb.predict(X_test)

# Точность
mae = mean_absolute_error(y_test, y_pred)
print("MAE лучшей модели:", mae)

Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 800, 'subsample': 0.8}
MAE лучшей модели: 1.154581098084783
